In [1]:
import pandas as pd
import numpy as np
from scipy import stats

df = pd.read_excel("customer_behavior.xlsx")
purchase = df["PurchaseAmount"].dropna()

### 1. What is the average, median, and mode of PurchaseAmount?

In [2]:
mean = purchase.mean()
median = purchase.median()
mode = purchase.mode().iloc[0]
print(f"Average PurchaseAmount: {mean:.2f}")
print(f"Median PurchaseAmount: {median:.2f}")
print(f"Mode PurchaseAmount: {mode:.2f}")

Average PurchaseAmount: 1003.95
Median PurchaseAmount: 998.08
Mode PurchaseAmount: 0.00


### 2. Are there any outliers in the PurchaseAmount data?

In [3]:
Q1 = purchase.quantile(0.25)
Q3 = purchase.quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR
outliers = purchase[(purchase < lower) | (purchase > upper)]
print(f"Lower bound: {lower:.2f}")
print(f"Upper bound: {upper:.2f}")
print(f"Number of outliers: {len(outliers)}")
print("Answer: Yes, there are outliers." if len(outliers) > 0 else "Answer: No outliers were detected.")

Lower bound: -306.51
Upper bound: 2307.23
Number of outliers: 15
Answer: Yes, there are outliers.


### 3. Is there any skewness or kurtosis in the PurchaseAmount distribution?

In [4]:
skewness = purchase.skew()
kurtosis = purchase.kurt()
print(f"Skewness: {skewness:.3f}")
print(f"Kurtosis: {kurtosis:.3f}")
print("Answer: The distribution is nearly symmetric with a slight positive skew and slightly negative excess kurtosis.")

Skewness: 0.106
Kurtosis: -0.262
Answer: The distribution is nearly symmetric with a slight positive skew and slightly negative excess kurtosis.


### 4. Is there a significant difference in spending between male and female customers?

In [5]:
male = df.loc[df["Gender"] == "Male", "PurchaseAmount"].dropna()
female = df.loc[df["Gender"] == "Female", "PurchaseAmount"].dropna()
t_stat, p_value = stats.ttest_ind(male, female, equal_var=False)
print(f"Male average: {male.mean():.2f}")
print(f"Female average: {female.mean():.2f}")
print(f"t-statistic: {t_stat:.3f}")
print(f"p-value: {p_value:.4f}")
print("Answer: There is a statistically significant difference in spending (p < 0.05); males spend slightly more on average." if p_value < 0.05 else "Answer: There is no statistically significant difference in spending (p >= 0.05).")

Male average: 1019.18
Female average: 987.87
t-statistic: 2.235
p-value: 0.0255
Answer: There is a statistically significant difference in spending (p < 0.05); males spend slightly more on average.


### 5. Is there a relationship between ProductCategory and customer churn?

In [6]:
table = pd.crosstab(df["ProductCategory"], df["Churn"])
chi2, p_value, dof, expected = stats.chi2_contingency(table)
print(table)
print(f"Chi-square statistic: {chi2:.3f}")
print(f"p-value: {p_value:.4f}")
print("Answer: There is a significant relationship between ProductCategory and churn." if p_value < 0.05 else "Answer: There is no statistically significant relationship between ProductCategory and churn.")

Churn             No  Yes
ProductCategory          
Electronics      785  541
Fashion          839  607
Grocery          855  604
Chi-square statistic: 0.396
p-value: 0.8204
Answer: There is no statistically significant relationship between ProductCategory and churn.


### 6. Does PurchaseAmount vary significantly across different regions?

In [7]:
region_summary = df.groupby("Region")["PurchaseAmount"].mean()
groups = [g["PurchaseAmount"].dropna().values for _, g in df.groupby("Region")]
f_stat, p_value = stats.f_oneway(*groups)
print(region_summary.round(2))
print(f"ANOVA F-statistic: {f_stat:.3f}")
print(f"p-value: {p_value:.4f}")
print("Answer: PurchaseAmount varies significantly across regions." if p_value < 0.05 else "Answer: PurchaseAmount does not vary significantly across regions.")

Region
East     1009.95
North    1013.02
South     997.60
West      995.25
Name: PurchaseAmount, dtype: float64
ANOVA F-statistic: 0.390
p-value: 0.7605
Answer: PurchaseAmount does not vary significantly across regions.


### 7. Which email campaign (A or B) performed better in terms of average PurchaseAmount?

In [8]:
campaign = df.groupby("CampaignGroup")["PurchaseAmount"].mean()
print(campaign.round(2))
best = campaign.idxmax()
print(f"Answer: Campaign {best} performed better, with the higher average PurchaseAmount of {campaign.max():.2f}.")

CampaignGroup
A    1011.95
B     994.34
Name: PurchaseAmount, dtype: float64
Answer: Campaign A performed better, with the higher average PurchaseAmount of 1011.95.


### 8. Can we assume PurchaseAmount follows a normal distribution?

In [9]:
statistic, p_value = stats.normaltest(purchase)
print(f"Normality test statistic: {statistic:.3f}")
print(f"p-value: {p_value:.6f}")
print("Answer: No. The normality test rejects the assumption of an exactly normal distribution (p < 0.05)." if p_value < 0.05 else "Answer: Yes. There is insufficient evidence to reject normality (p >= 0.05).")

Normality test statistic: 27.252
p-value: 0.000001
Answer: No. The normality test rejects the assumption of an exactly normal distribution (p < 0.05).


### 9. What insights can we gain by applying the Central Limit Theorem?

In [10]:
sample_size = 30
number_of_samples = 1000
rng = np.random.default_rng(42)
sample_means = [rng.choice(purchase.to_numpy(), size=sample_size, replace=True).mean() for _ in range(number_of_samples)]
print(f"Population mean: {purchase.mean():.2f}")
print(f"Mean of sample means: {np.mean(sample_means):.2f}")
print(f"Standard deviation of sample means: {np.std(sample_means, ddof=1):.2f}")
print("Answer: By the Central Limit Theorem, the distribution of sample means becomes approximately normal for sufficiently large samples, even when the original PurchaseAmount distribution is not perfectly normal. This allows reliable estimation and confidence intervals for the population mean.")

Population mean: 1003.95
Mean of sample means: 1002.11
Standard deviation of sample means: 87.79
Answer: By the Central Limit Theorem, the distribution of sample means becomes approximately normal for sufficiently large samples, even when the original PurchaseAmount distribution is not perfectly normal. This allows reliable estimation and confidence intervals for the population mean.


### 10. What is the 95% confidence interval for the average PurchaseAmount?

In [11]:
mean = purchase.mean()
sem = stats.sem(purchase)
ci = stats.t.interval(0.95, df=len(purchase)-1, loc=mean, scale=sem)
print(f"Average PurchaseAmount: {mean:.2f}")
print(f"95% Confidence Interval: ({ci[0]:.2f}, {ci[1]:.2f})")
print(f"Answer: We are 95% confident that the population average PurchaseAmount lies between {ci[0]:.2f} and {ci[1]:.2f}.")

Average PurchaseAmount: 1003.95
95% Confidence Interval: (990.38, 1017.52)
Answer: We are 95% confident that the population average PurchaseAmount lies between 990.38 and 1017.52.
